In [1]:
# Milvus vector database imports
import pymilvus
from pymilvus import MilvusClient  # High-level client for simple operations
from pymilvus import (
    utility,                        # Utility functions for Milvus
    FieldSchema,                    # Define schema fields
    CollectionSchema,               # Define collection structure
    DataType,                       # Data types for fields
    Collection,                     # Low-level collection operations
    AnnSearchRequest,               # Create search requests for hybrid search
    RRFRanker,                      # Reciprocal Rank Fusion reranker
    connections,                    # Manage Milvus connections
)

# Sparse embedding import 
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer
from pymilvus.model.sparse import BM25EmbeddingFunction , SpladeEmbeddingFunction

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datasets import load_dataset

sns.set_style("whitegrid")
plt.rcParams['figure.figsize']=(12,6)


/Users/nilasark/advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("tdiggelm/climate_fever")
corpus_text=dataset['test']['claim']
for i,claim in enumerate(corpus_text[:5],1):
    print(f"{i}:{claim}")
print(f"\n Loaded total {len(corpus_text)} claims from the dataset")

# ============================================================
# Build and Fit BM25 Embedding Function
# ============================================================

# Build a default English language analyzer (tokenizer + stemmer)
# This will break text into tokens and normalize them (e.g., "running" → "run")
analyzer=build_default_analyzer(language='en')
bm25_ef=BM25EmbeddingFunction(analyzer)


# Create BM25 embedding function with the analyzer
# BM25 uses statistical Term Frequency-Inverse Document Frequency (TF-IDF)
print(f"fitting BM25 on conpus")
bm25_ef.fit(corpus_text)
print(f"\nBM25 fitted Vocabolary size:{bm25_ef.dim}" )


1:Global warming is driving polar bears toward extinction
2:The sun has gone into ‘lockdown’ which could cause freezing weather, earthquakes and famine, say scientists
3:The polar bear population has been growing.
4:Ironic' study finds more CO2 has slightly cooled the planet
5:Human additions of CO2 are in the margin of error of current measurements and the gradual increase in CO2 is mainly from oceans degassing as the planet slowly emerges from the last ice age.

 Loaded total 1535 claims from the dataset
fitting BM25 on conpus

BM25 fitted Vocabolary size:3410


In [3]:
documents = [
    "Currently, scientists none of these places, which today supply much of the world's food, will be reliable sources of any.",
    "Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.",
    "The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker.",
    "Coral becomes stressed and expels the algae, which leave the coral a bleached white color.",
    "The rapid changes in the climate may have profound consequences for humans and other species... Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting in intense and widespread forest fires in Indonesia that belched out a vast quantity of greenhouse gas"
]

print(f"Prepared {len(documents)} from embedding\n")

print(f"Creating BM25 embeddings")
bm25_embeddings_docs=bm25_ef.encode_documents(documents)

print(f"Sparse Vector Dimension: {bm25_ef.dim}")
print(f"{list(bm25_embeddings_docs)[0].shape}")

print(f"\nCreating Splade Embeddings")
splade_ef=SpladeEmbeddingFunction(
    model_name="naver/splade-v3",
    device="cpu"
)
splade_embedding_docs=splade_ef.encode_documents(documents)
print(f"SPLADE Vector Dimension: {splade_ef.dim}")
print(f"Splade Vector Shape:{list(splade_embedding_docs)[0].shape}")

Prepared 5 from embedding

Creating BM25 embeddings
Sparse Vector Dimension: 3410
(3410,)

Creating Splade Embeddings
SPLADE Vector Dimension: 30522
Splade Vector Shape:(30522,)
